# Part 3: Metrics + Threshold — Where to Set the Operating Point
**⏱ This section takes approximately 30 minutes.**

---

## Scenario: Thursday — The Accuracy is a Lie

By Wednesday evening Sarah had a trained logistic regression with cross-validated accuracy of 88%. Today she's nervous, because:

> *"Accuracy 88% sounds great. But the baseline — guessing 'stayed' for everyone — is also 88%. So either the model is doing nothing useful, or accuracy is hiding the truth."*

She's about to discover the truth, and it'll change everything about what she presents to Marcus on Friday.

---

## The objective for today

Sarah needs to answer one specific business question before Friday:

> **Which customers should the retention team call next week?**

NorthStar's retention team can handle about **200 calls per week**. That capacity constraint — not any single metric — is what determines the right operating point. Sarah's job is to pick the threshold that surfaces roughly 200 customers, maximises the fraction of real churners caught, and lets her explain the tradeoffs honestly to Marcus.

This notebook is the path from "88% accuracy" to that defensible, business-aware recommendation.

---

**By the end of this notebook you will be able to:**
- Read a confusion matrix and explain what each cell means in business terms
- Define and compute precision, recall, and F1 — and know when each matters
- Explain why accuracy is a misleading metric on imbalanced data
- Understand the precision–recall tradeoff: why you can't maximise both at once
- Pick a classification threshold that matches a business constraint (capacity, cost), not just the default 0.5

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, precision_recall_curve,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)

print("✅ Libraries loaded — including sklearn.metrics")

## Step 1 — Rebuild the trained pipeline

(Same code path as yesterday. We re-fit so this notebook is self-contained.)

In [ ]:
df = pd.read_csv("data/northstar_churn.csv")
y  = df["churned"]
X  = df.drop(columns=["customer_id", "churned"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42,
)

numeric_features = ["age", "tenure_months", "num_purchases_quarter",
                    "avg_monthly_spend_gbp", "returns_per_purchase",
                    "last_login_days_ago", "avg_review_polarity",
                    "support_tickets_quarter"]
categorical_features = ["region", "subscription_tier"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features),
])

full_pipeline = Pipeline([
    ("prep",  preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
full_pipeline.fit(X_train, y_train)

print(f"Pipeline ready.  Test-set accuracy: {full_pipeline.score(X_test, y_test):.3f}")

## Step 2 — From probabilities to predictions: the threshold

### What logistic regression actually outputs

In yesterday's notebook we trained a logistic regression. Under the hood it didn't just output "churn / stayed" — it output a **probability between 0 and 1** for every customer via the sigmoid function:

$$\hat{p} = \sigma(z) = \frac{1}{1 + e^{-z}} \qquad \text{where } z = w_0 + w_1 x_1 + \cdots + w_n x_n$$

The model is saying: *"I think this customer has a 34% chance of churning."*

### What the threshold does

To convert that probability into a binary label (churn / stayed), sklearn applies a **threshold**:

$$\text{prediction} = \begin{cases} 1 \text{ (churn)} & \text{if } \hat{p} \geq \text{threshold} \\ 0 \text{ (stayed)} & \text{if } \hat{p} < \text{threshold} \end{cases}$$

`.predict()` uses threshold = **0.5** by default. That means a customer needs to be *more likely than not* to churn before the model flags them. On a dataset where only 12% of customers churn, this is a very high bar — and the root cause of the problem Sarah suspects.

`.predict_proba()` returns the raw probabilities, before any threshold is applied. We'll work with those directly so we can **move the threshold** and see what changes.

In [ ]:
# Probabilities of class 1 (= churned)
y_proba = full_pipeline.predict_proba(X_test)[:, 1]
y_pred_default = (y_proba >= 0.5).astype(int)

print(f"y_proba shape: {y_proba.shape}")
print()
print("Sample predicted probabilities for the first 10 test customers:")
sample = pd.DataFrame({
    "customer_idx": X_test.index[:10],
    "actual":       y_test.iloc[:10].values,
    "prob(churn)":  y_proba[:10].round(3),
    "label@0.5":    y_pred_default[:10],
})
print(sample.to_string(index=False))

## Step 3 — The confusion matrix and the four metrics

### The four cells

Here is the moment of truth. Group every test-set prediction into one of four bins:

|  | Predicted **STAYED** | Predicted **CHURN** |
|---|---|---|
| **Actually STAYED** | TN — correctly let stayer alone | FP — wasted intervention |
| **Actually CHURN** | FN — missed real churner | TP — caught a real churner |

### Why accuracy isn't enough — and what the other metrics mean

From those four cells we can compute four metrics. Each asks a different question:

| Metric | Formula | Business question it answers |
|---|---|---|
| **Accuracy** | $\frac{TP + TN}{TP + TN + FP + FN}$ | Of all customers, what fraction did we label correctly? |
| **Precision** | $\frac{TP}{TP + FP}$ | Of all customers we *flagged*, what fraction were actually churners? |
| **Recall** | $\frac{TP}{TP + FN}$ | Of all customers who *actually churned*, what fraction did we catch? |
| **F1** | $\frac{2 \times \text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$ | Balanced average of precision and recall (useful when both errors hurt) |

**Why accuracy fails here:** on NorthStar's dataset, 88% of customers *stayed*. A model that predicts "stayed" for everyone gets 88% accuracy — without catching a single churner. Accuracy rewards the easy majority class. Precision and recall force us to look at what the model does on the minority class (churners) specifically.

**The precision–recall intuition:**
- **High precision, low recall** → the model is *conservative*: it only flags customers it's very confident about, but it misses many real churners.
- **High recall, low precision** → the model is *aggressive*: it flags many customers, catching most real churners, but also flags a lot of stayers by mistake.

The right balance depends on the business. We'll see exactly how to choose it in Steps 4 and 5.

In [ ]:
cm = confusion_matrix(y_test, y_pred_default)
tn, fp, fn, tp = cm.ravel()

print(f"Confusion matrix @ threshold = 0.5:")
print(f"                    pred=stayed   pred=churn")
print(f"actual=stayed       TN = {tn:>4}    FP = {fp:>4}")
print(f"actual=churn        FN = {fn:>4}    TP = {tp:>4}")
print()

# Visualise
fig, ax = plt.subplots(figsize=(6, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=["stayed", "churn"])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion matrix @ threshold = 0.5")
plt.tight_layout()
plt.show()

# Compute the four headline metrics
acc  = accuracy_score(y_test, y_pred_default)
prec = precision_score(y_test, y_pred_default, zero_division=0)
rec  = recall_score(y_test, y_pred_default, zero_division=0)
f1   = f1_score(y_test, y_pred_default, zero_division=0)

print(f"Metrics @ 0.5:")
print(f"  Accuracy:   {acc:.3f}")
print(f"  Precision:  {prec:.3f}   (of all flagged, fraction actually churners)")
print(f"  Recall:     {rec:.3f}   (of all actual churners, fraction we flagged)")
print(f"  F1:         {f1:.3f}")

### 💡 What the numbers at threshold 0.5 tell us

Apply the definitions above to what you just saw:

- **Accuracy ≈ 88%** — looks great. But recall that accuracy = (TP + TN) / all. With 88% stayers in the dataset, a model that predicts "stayed" for everyone would also score 88%. Accuracy cannot distinguish a useful model from a useless one here.
- **Recall is very low** — recall = TP / (TP + FN). At threshold 0.5, FN (missed churners) is very large. The model is catching only a small fraction of actual churners. Most real churners slip through as false negatives.
- **Precision is OK** — precision = TP / (TP + FP). When the model does say "churn", it's usually right. But it almost never says "churn", so this is cold comfort.

**What's happening:** at threshold 0.5, the model is being *conservative*. With only 12% of customers actually churning, the model needs to be more than 50% confident before it flags anyone. That's a high bar for a rare event.

**Sarah's reaction:** *"If I show Marcus 88% accuracy, he'll be pleased. Then a churner will leave, he'll ask why we didn't flag them, and I'll have nothing to say. I need to lower the threshold — accept more false positives in exchange for catching more real churners."*

This is the **precision–recall tradeoff in action.** The next step is to see what happens when we move the threshold.

## ⏸️ Pause and Predict

Before we sweep across thresholds, predict:

- If we lower the threshold from 0.5 to 0.3 (flag anyone with > 30% predicted churn probability), what happens to:
  - **Recall** — will it go up, down, or stay the same?
  - **Precision** — same question?
  - **Accuracy** — same question?
- What's the *minimum* recall Sarah should accept before walking into Marcus's office?

*Write your predictions here:*

> *Sample answers:*
> - **Recall ↑** — we'll flag more customers, including more actual churners. Recall climbs.
> - **Precision ↓** — but a bigger fraction of the additional flags will be false positives. Precision drops.
> - **Accuracy ↓** — when the model says "churn" more, it's wrong more often on stayers. Accuracy slips because the data is so imbalanced.
> - **Minimum acceptable recall** — context-dependent, but anything below 50% means we're missing more than we catch. For an "early warning" system, 70% is a reasonable target.

## Step 4 — Sweep across thresholds

Let's see precision, recall, and F1 as functions of the threshold. The right operating point depends on which trade-off Sarah is willing to make.

In [ ]:
thresholds = np.arange(0.05, 0.95, 0.02)
records = []

for t in thresholds:
    pred = (y_proba >= t).astype(int)
    if pred.sum() == 0:
        records.append((t, np.nan, 0.0, np.nan, pred.sum()))
        continue
    records.append((
        t,
        precision_score(y_test, pred, zero_division=0),
        recall_score(y_test, pred, zero_division=0),
        f1_score(y_test, pred, zero_division=0),
        pred.sum(),
    ))

sweep = pd.DataFrame(records, columns=["threshold", "precision", "recall", "F1", "n_flagged"])

# Plot
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(sweep["threshold"], sweep["precision"], label="Precision", linewidth=2, color="steelblue")
ax.plot(sweep["threshold"], sweep["recall"],    label="Recall",    linewidth=2, color="coral")
ax.plot(sweep["threshold"], sweep["F1"],        label="F1",        linewidth=2, color="seagreen", linestyle="--")
ax.axvline(0.5, color="gray", linestyle=":", linewidth=1, label="Default threshold (0.5)")
ax.axvline(0.3, color="indianred", linestyle=":", linewidth=1.5, label="Sarah's recommendation (0.3)")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision / Recall / F1 as the threshold changes")
ax.legend(loc="lower left")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print("Threshold      Precision    Recall      F1     N flagged")
for t in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
    row = sweep.iloc[(sweep["threshold"] - t).abs().idxmin()]
    print(f"  {row['threshold']:.2f}         {row['precision']:.3f}      {row['recall']:.3f}     {row['F1']:.3f}     {int(row['n_flagged'])}")

### 💡 The precision–recall tradeoff made visible

Read the table column by column to see the tradeoff play out:

- **Lowering the threshold** (e.g. 0.5 → 0.3) flags more customers:
  - **Recall rises** — we catch more of the real churners, because we're now flagging anyone with > 30% predicted probability instead of > 50%.
  - **Precision falls** — more of the additional flags are stayers who merely *looked* risky. Our hit rate on the flagged list drops.
  - **F1 first rises then falls** — it balances both. Its peak is the threshold that most evenly trades the two errors.
  - **Accuracy falls** — every time we flag a stayer (FP), accuracy ticks down. But since stayers are 88% of the data, accuracy was never measuring what we cared about.

- **Raising the threshold** (e.g. 0.5 → 0.7) is the mirror image:
  - Precision rises (only very confident predictions get flagged), but recall collapses.

**This tradeoff is not a flaw — it is a fundamental property of classification.** There is no threshold that maximises both precision and recall simultaneously. Every threshold represents a choice about which error to minimise.

**F1 peaks somewhere between 0.2 and 0.4** — the threshold that balances the two errors. That is a useful default when you have no other guidance.

**But Sarah has better guidance than F1.** Her retention team has a capacity limit — they can only call ~200 customers per week. That constraint is a more concrete anchor than any single metric. Step 5 will show how to use it.

## Step 5 — Make the business-aware choice

### Why the "best" threshold isn't a metric question

Step 4 showed that every threshold is a tradeoff: higher recall always costs precision, and vice versa. So how do you decide WHERE to land?

Real-world constraints break the tie:

| Constraint | What it means for the threshold |
|---|---|
| **Team capacity** | The team can only act on N flagged customers per week. Set the threshold so the flag volume ≈ N. |
| **Relative error cost** | If missing a churner (FN) is far more expensive than a wasted call (FP), lower the threshold to prioritise recall. |
| **Precision floor** | If retention calls are costly or annoy customers, you may need precision ≥ X% — set threshold accordingly. |

**At NorthStar**, the binding constraint is capacity: the retention team can call **~200 customers per week** from the test-set's worth of data (≈ 10% of the full customer base). Marcus will not thank Sarah for a threshold that surfaces 800 customers — the team simply cannot work through that volume.

Sarah's goal: find the threshold where `n_flagged ≈ 200`, then check whether the resulting recall is high enough to be operationally useful — and be able to explain the precision tradeoff she accepted to get there.

In [ ]:
# Pick the threshold so the number flagged is approximately the team's capacity
target_capacity = 200  # in the test set
best_threshold = sweep.iloc[(sweep["n_flagged"] - target_capacity).abs().idxmin()]["threshold"]

# Apply
y_pred_business = (y_proba >= best_threshold).astype(int)

cm_b = confusion_matrix(y_test, y_pred_business)
tn_b, fp_b, fn_b, tp_b = cm_b.ravel()

print(f"Chosen threshold: {best_threshold:.2f}")
print(f"Number flagged:   {int(y_pred_business.sum())}  (team capacity ~ {target_capacity})")
print()
print(f"Confusion matrix @ threshold {best_threshold:.2f}:")
print(f"                    pred=stayed   pred=churn")
print(f"actual=stayed       TN = {tn_b:>4}    FP = {fp_b:>4}")
print(f"actual=churn        FN = {fn_b:>4}    TP = {tp_b:>4}")
print()

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, t, p in zip(axes, [0.5, best_threshold], [y_pred_default, y_pred_business]):
    cm_x = confusion_matrix(y_test, p)
    ConfusionMatrixDisplay(cm_x, display_labels=["stayed", "churn"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d",
    )
    ax.set_title(f"Threshold = {t:.2f}")

plt.tight_layout()
plt.show()

# Compare metrics
print(f"\nDecision summary:")
print(f"{'':22s}  threshold=0.5    threshold={best_threshold:.2f}")
for metric_fn, name in [(accuracy_score,  "Accuracy"),
                         (precision_score, "Precision"),
                         (recall_score,    "Recall"),
                         (f1_score,        "F1")]:
    kwargs = {"zero_division": 0} if name != "Accuracy" else {}
    v1 = metric_fn(y_test, y_pred_default,  **kwargs)
    v2 = metric_fn(y_test, y_pred_business, **kwargs)
    arrow = "↑" if v2 > v1 else ("↓" if v2 < v1 else "→")
    print(f"  {name:20s}      {v1:.3f}             {v2:.3f}  {arrow}")

### 💡 What this tells us — and what Sarah will say to Marcus

- **At threshold 0.5** — the model surfaces only a handful of customers. It "looks accurate" (88%) but is operationally useless: it catches barely any real churners.
- **At threshold ~0.3** — Sarah surfaces ~200 customers (matching the retention team's capacity). Recall climbs substantially — she's catching a much higher fraction of real churners. The tradeoff: precision drops, meaning some of those 200 flagged customers wouldn't have churned anyway. That is the cost of getting recall up.
- **Accuracy went DOWN** when we lowered the threshold — but accuracy was the wrong metric all along.

**The tradeoff Sarah accepts:**
- She gains: **higher recall** — most real churners will be caught.
- She accepts: **lower precision** — some of the 200 calls will be "wasted" on customers who weren't actually going to churn.
- Whether that tradeoff is worth it depends on the cost of a missed churner vs the cost of a wasted retention call — which leads directly to cost-sensitive thresholding (Extension 3 of this notebook).

**Back to our scenario:**
> Sarah's recommendation to Marcus: *"Use the model at threshold 0.3. We flag about 200 customers a week — that fits your team's capacity. We catch the majority of the real churners. We accept that some of those flagged customers wouldn't have left anyway — that's the cost of keeping recall high."*

**What Marcus will ask next:** *"How much money does this actually save?"* That cost–benefit analysis is in **Extension 3** of this notebook — it puts a £ value on each false positive and false negative, and finds the cost-minimising threshold.

## ✅ Section Summary

| What we learned | Key takeaway |
|---|---|
| **Accuracy is misleading** on imbalanced datasets — `88%` was just `1 - churn_rate` | Always pair accuracy with precision/recall |
| **Confusion matrix** tells you exactly where the model gets things wrong (FP vs FN) | Read it before trusting any single metric |
| **Precision vs recall** is a tradeoff — choose based on business cost of each error | No "right answer" without context |
| **F1** balances precision and recall — useful when both errors hurt | Reach for this when in doubt |
| **Threshold choice** is a business decision, not a technical one | Capacity + cost dictate where to set it |

---

## 🏁 Friday — What Sarah Presents to Marcus

| Item | Number |
|---|---|
| **Model** | One sklearn `Pipeline` — preprocess + logistic regression. Same code path at training and prediction. |
| **Cross-validated accuracy** | 88% (but irrelevant — see below) |
| **Recommended operating threshold** | **0.30** — matches retention team capacity (~200 customers/week) |
| **At threshold 0.30** | Recall higher than at 0.5; precision lower but still actionable |
| **Top predictive features** | High returns, low review polarity, short tenure, non-premium tier |

Marcus listens. He nods. Then he asks:

> *"Sarah — this is the first model we own. Can you make it BETTER next week? We're hitting capacity. If you could squeeze more recall without more false positives, we'd save real money. Try those tree-based models you mentioned."*

That question — **"can you make it better?"** — is the engine of **L04 (Advanced Supervised Learning).**

---

**Next step →** Open `assignment.ipynb` to apply this pipeline to a new domain (Lakeside Bank).

*Or, for deeper material on bias-variance math, ROC-AUC, learning curves, and manual feature engineering: open `optional_extensions.ipynb`.*

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## Extension 1 — The precision-recall curve

The sklearn `precision_recall_curve` function returns the precision and recall at every possible threshold. Plot precision vs recall to see the *frontier* of possible operating points.

In [ ]:
prec_curve, rec_curve, thr_curve = precision_recall_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(rec_curve, prec_curve, linewidth=2, color="steelblue")
ax.fill_between(rec_curve, prec_curve, alpha=0.1, color="steelblue")
ax.axhline(y_test.mean(), color="gray", linestyle="--", linewidth=1,
           label=f"Random baseline (= class rate = {y_test.mean():.2f})")

# Mark the two thresholds
for t, label, color in [(0.5, "T=0.5", "navy"), (0.3, "T=0.3", "indianred")]:
    pred = (y_proba >= t).astype(int)
    p_t  = precision_score(y_test, pred, zero_division=0)
    r_t  = recall_score(y_test, pred, zero_division=0)
    ax.scatter([r_t], [p_t], color=color, s=120, zorder=5, edgecolor="white", linewidth=2)
    ax.annotate(label, (r_t, p_t), xytext=(10, 10), textcoords="offset points",
                fontweight="bold", fontsize=12, color=color)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision–Recall Curve (test set)")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print("The PR curve traces the achievable (precision, recall) pairs as the threshold varies.")
print("Operating closer to the top-right = better model. Below the dashed line = worse than random.")

## Extension 2 — F1 vs threshold — finding the maximiser

If you don't have an explicit capacity constraint, F1 is the natural metric to maximise. Plot F1 against threshold and pick the peak.

In [ ]:
best_idx = sweep["F1"].idxmax()
best_row = sweep.iloc[best_idx]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(sweep["threshold"], sweep["F1"], linewidth=2, color="seagreen")
ax.axvline(best_row["threshold"], color="indianred", linestyle="--",
           label=f"Max F1 = {best_row['F1']:.3f} at threshold = {best_row['threshold']:.2f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("F1 score")
ax.set_title("F1 score across thresholds — the peak is the 'balanced' choice")
ax.legend()
ax.set_ylim(0, max(0.5, best_row["F1"] * 1.2))
plt.tight_layout()
plt.show()

print(f"F1-optimal threshold: {best_row['threshold']:.2f}")
print(f"At that threshold:    precision = {best_row['precision']:.3f}, recall = {best_row['recall']:.3f}, F1 = {best_row['F1']:.3f}")

## Extension 3 — Cost-sensitive thresholding with capacity constraint

If you can put a £ value on each type of error, you can find the threshold that *minimises expected cost*.

**NorthStar's numbers:**
- FP cost: £20 per wasted retention call (analyst time + discount offered)
- FN cost: £150 per lost customer (approximated lifetime value)

The cost function for any threshold is: `total_cost = 20·FP + 150·FN`

But **cost alone isn't the full story** — the team still has a capacity limit. We'll look at two scenarios:

1. **Unconstrained** — the pure cost-minimising threshold, ignoring flag volume
2. **Capacity-constrained** — the best threshold that also respects the ~200-calls-per-week limit

Notice that these two answers can differ: if the cost-optimal threshold flags 600 customers but the team can only handle 200, the unconstrained answer is operationally useless.

In [ ]:
COST_FP = 20.0    # £ per wasted call
COST_FN = 150.0   # £ per lost customer
CAPACITY = 200    # max customers the team can call

costs = []
for t in np.arange(0.05, 0.95, 0.01):
    pred = (y_proba >= t).astype(int)
    cm_t = confusion_matrix(y_test, pred)
    _, fp, fn, _ = cm_t.ravel()
    n_flagged = int(pred.sum())
    cost = COST_FP * fp + COST_FN * fn
    costs.append((t, fp, fn, n_flagged, cost))

cost_df = pd.DataFrame(costs, columns=["threshold", "FP", "FN", "n_flagged", "expected_cost_gbp"])

# --- Scenario 1: pure cost minimum (no capacity constraint) ---
best_unconstrained = cost_df.iloc[cost_df["expected_cost_gbp"].idxmin()]

# --- Scenario 2: capacity-constrained cost minimum ---
within_capacity = cost_df[cost_df["n_flagged"] <= CAPACITY]
best_constrained = within_capacity.iloc[within_capacity["expected_cost_gbp"].idxmin()]

# Plot: cost curve with both thresholds marked
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df_subset, title in [
    (axes[0], cost_df, "Unconstrained — full threshold range"),
    (axes[1], within_capacity, f"Capacity-constrained — n_flagged ≤ {CAPACITY}"),
]:
    ax.plot(df_subset["threshold"], df_subset["expected_cost_gbp"], linewidth=2, color="indianred")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Expected total cost (£)")
    ax.set_title(title)

axes[0].axvline(best_unconstrained["threshold"], color="navy", linestyle="--",
                label=f"Min cost = £{best_unconstrained['expected_cost_gbp']:.0f} "
                      f"at T = {best_unconstrained['threshold']:.2f} "
                      f"({int(best_unconstrained['n_flagged'])} flags)")
axes[0].legend(fontsize=9)

axes[1].axvline(best_constrained["threshold"], color="seagreen", linestyle="--",
                label=f"Min cost within capacity = £{best_constrained['expected_cost_gbp']:.0f} "
                      f"at T = {best_constrained['threshold']:.2f} "
                      f"({int(best_constrained['n_flagged'])} flags)")
axes[1].legend(fontsize=9)

plt.suptitle(f"Cost-optimal threshold  (FP cost = £{COST_FP:.0f} · FN cost = £{COST_FN:.0f} · capacity = {CAPACITY})",
             fontsize=12)
plt.tight_layout()
plt.show()

print("=" * 60)
print(f"Scenario 1 — UNCONSTRAINED cost minimum:")
print(f"  Threshold:    {best_unconstrained['threshold']:.2f}")
print(f"  Flags:        {int(best_unconstrained['n_flagged'])}  (team capacity is {CAPACITY})")
print(f"  FP / FN:      {int(best_unconstrained['FP'])} / {int(best_unconstrained['FN'])}")
print(f"  Total cost:   £{best_unconstrained['expected_cost_gbp']:.0f}")
print()
print(f"Scenario 2 — CAPACITY-CONSTRAINED cost minimum (≤ {CAPACITY} flags):")
print(f"  Threshold:    {best_constrained['threshold']:.2f}")
print(f"  Flags:        {int(best_constrained['n_flagged'])}")
print(f"  FP / FN:      {int(best_constrained['FP'])} / {int(best_constrained['FN'])}")
print(f"  Total cost:   £{best_constrained['expected_cost_gbp']:.0f}")
print()
print("The constrained cost is higher — capacity prevents catching some churners —")
print("but it reflects what the team can actually deliver this week.")